In [ ]:
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
var myBatch = ExecutionQueues[1];
myBatch

RuntimeLocation,AdditionalEnvironmentVars,DeploymentBaseDirectory,DeployRuntime,Name,DotnetRuntime,Username,NumOfAdditionalServiceCores,NumOfAdditionalServiceCoresMPISerial,NumOfServiceCoresPerMPIprocess,ServerName,ComputeNodes,DefaultJobPriority,SingleNode,AllowedDatabasesPaths
win\amd64,[ ],\\dc3\userspace\smuda\hpccluster\binaries,True,FDY-WindowsHPC,dotnet,FDY\smuda,0,0,0,DC3,<null>,Normal,True,[ \\dc3\userspace\smuda\hpccluster\ ]


In [ ]:
BoSSSshell.WorkflowMgm.Init("RisingBubble3D", myBatch);

Project name is set to 'RisingBubble3D'.
Opening existing database '\\dc3\userspace\smuda\hpccluster\RisingBubble3D'.


In [ ]:
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

In [ ]:
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Application.XNSE_Solver.Logging;

## Sessions

In [ ]:
bool restartRun = true;

In [ ]:
var sessions = BoSSSshell.WorkflowMgm.Sessions;
sessions

#0: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_Reinit30_BDF3_singleCore_testcase1*	09/10/2024 12:38:07	cf93bc52...
#1: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_Reinit30_BDF3_fixedInterfaceCheck_testcase1*	09/05/2024 15:13:05	00b7639e...
#2: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_Reinit30_BDF3_testcase1*	09/05/2024 08:59:21	d25fc934...
#3: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_testcase1_withPreEnforcerActive_restart1*	08/28/2024 15:41:52	a929ee94...
#4: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_testcase1*	08/09/2024 11:54:11	cb26cb13...
#5: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_testcase1_restart1*	08/05/2024 09:45:57	9d1f82be...
#6: RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_testcase1_setttingsCheck2	08/01/2024 09:27:13	eff0e8c8...
#7: RisingBubble3D	RB3D_16x16x32AMR1_k2_testcase1_setttingsCheck	04/19/2024 16:25:45	19291211...


In [ ]:
//sessions.Pick(0).Delete(true); //.Export().WithSupersampling(2).Do()

In [ ]:
var restartSession = sessions.Pick(2);
restartSession

RisingBubble3D	RB3D_fullDomain_8x8x16AMR0_k3_Reinit30_BDF3_testcase1*	09/05/2024 08:59:21	d25fc934...

In [ ]:
//restartSession.Timesteps.Skip(0).Export().WithSupersampling(3).Do()

In [ ]:
string restartName = "RB3D_fullDomain_8x8x16AMR0_k3_Reinit30_BDF3_testcase1_withOutputs_restart1";

## Grid Creation

In [ ]:
bool fullDomain = true;

In [ ]:
int[] Resolutions = new int[] { 8 };
IGridInfo[] Grids = new IGridInfo[Resolutions.Length];

for(int i = 0; i < Resolutions.Length; i++) {
    int Res = Resolutions[i];
    string GridName = $"RisingBubble3D_{Res}x{Res}x{2*Res}";
    GridName = fullDomain ? GridName + "_fullDomain" : "_quarterDomain";

    IGridInfo cachedGrid = wmg.Grids.FirstOrDefault(grid => grid.Name == GridName);
    cachedGrid = null;
    if(cachedGrid == null) {
        
        // must create new Grid
        double[] xNodes = fullDomain ? GenericBlas.Linspace(0, 1.0, Res + 1) : GenericBlas.Linspace(0.5, 1.0, Res + 1);
        double[] yNodes = xNodes;
        double[] zNodes = GenericBlas.Linspace(0, 2.0, Res*2 + 1);
        
        var grd = Grid3D.Cartesian3DGrid(xNodes, yNodes, zNodes);
        grd.Name = GridName;
        
        grd.DefineEdgeTags(delegate(Vector X) {
            string ret = null;

            if (fullDomain) {
                if(X.x.Abs() <= 1e-8 || X.y.Abs() <= 1.0e-8)
                ret = IncompressibleBcType.Wall.ToString();
            } else {
                if((X.x - 0.5).Abs() <= 1e-8 || (X.y - 0.5).Abs() <= 1.0e-8)
                ret = IncompressibleBcType.SlipSymmetry.ToString();
            }
            if((X.x - 1.0).Abs() <= 1e-8 || (X.y - 1.0).Abs() <= 1.0e-8)
                ret = IncompressibleBcType.Wall.ToString();
            if((X.z).Abs() <= 1e-8)         // lower wall
                ret = IncompressibleBcType.Wall.ToString();
            if((X.z - 2.0).Abs() <= 1e-8)   // upper wall
                ret = IncompressibleBcType.Wall.ToString();

            return ret;
        });    
        
        // grd.AddPredefinedPartitioning("FourProcSplit_fullDomain", delegate (double[] X) {
        //     int rank;
        //     double x = X[0]; double y = X[1];
        //     if (x < 0.5 && y < 0.5)
        //         rank = 0;
        //     else if (x >= 0.5 && y < 0.5)
        //         rank = 1;
        //     else if (x < 0.5 && y >= 0.5)
        //         rank = 2;
        //     else 
        //         rank = 3;

        //     return rank;
        // });
        
        Grids[i] = wmg.SaveGrid(grd);
        
    } else {
        //Console.WriteLine($"type: {cachedGrid.GetType()}, is IGridInfo? {cachedGrid is IGridInfo}");
        Console.WriteLine("Grid already found in database - identifid by name " + GridName);
        Grids[i] = cachedGrid;
    }
    
}

Opening existing database 'D:\local\RisingBubble3D'.
Opening existing database '\\wsl.localhost\Ubuntu\home\smuda\lichtb_scratch\bosss_databases\RisingBubble3D'.
Grid Edge Tags changed.
An equivalent grid (1a4702c1-21b5-433b-99b1-87acbc014d40) is already present in the database -- the grid will not be saved.


## Initial Values

In [ ]:
var PhiFunc = new Formula(
    "Phi",
    false,
    "using ilPSP.Utils; " + 
    "double dropRadius = 0.25;" + 
    "double Phi(double[] X) { " + 
    "    return Math.Sqrt((X[0] - 0.5).Pow2() + (X[1] - 0.5).Pow2() + (X[2] - 0.5).Pow2()) - dropRadius; " + 
    "}");

In [ ]:
var GravityValue = new Formula(
    "grav",
    false,
    "using ilPSP.Utils; " + 
    "double grav(double[] X) { return -0.98; }");

## Physical Settings

In [ ]:
string testcase = "testcase1";

In [ ]:
double rhoA = 1.0;      // bubble
double rhoB = 1.0;   // surrounding fluid
double muA = 1.0;
double muB = 1.0;
double sigma = 0.0;

switch(testcase) {
    case "testcase1": {
        rhoA = 100.0;
        rhoB = 1000.0;
        muA = 1.0;
        muB = 10.0;
        sigma = 24.5;

        break;
    }
    case "testcase2": {
        rhoA = 1.0;
        rhoB = 1000.0;
        muA = 0.1;
        muB = 10.0;
        sigma = 1.96;
        
        break;
    }
}


In [ ]:
int k = 3;
int AMRlevel = 0;
double hmin = (1.0 / (double)Resolutions[0]) / ((double)AMRlevel + 1);
double dt_sigma = BoSSS.Solution.XNSECommon.XNSEUtils.GetCapillaryTimeStep(rhoA, rhoB, sigma, hmin, k+1);
dt_sigma

0.01056655405839857

In [ ]:
double dt = 0.01;

In [ ]:
int numTs = (int)(3.0/dt);
numTs

300

## Control settings

In [ ]:
List<XNSE_Control> Controls = new List<XNSE_Control>();

In [ ]:
restartSession.Timesteps.Last()

 { Time-step: 73; Physical time: 0.7300000000000004s; Fields: Phi, PhiDG, VelocityX, VelocityY, VelocityZ, Pressure, Residual-MomentumX, Residual-MomentumY, Residual-MomentumZ, Residual-ContiEq, Velocity0X_Mean, Velocity0Y_Mean, Velocity0Z_Mean, GravityZ#A, GravityZ#B, VelocityX@Phi, VelocityY@Phi, VelocityZ@Phi, MPIrank, CutCells; Name:  }

In [ ]:
int restartTS = restartSession.Timesteps.Last().TimeStepNumber.MajorNumber - 0;
restartTS

73

In [ ]:
for (int idx = 0; idx < Grids.Length; idx++) {

    XNSE_Control C = new XNSE_Control();

    C.SetDGdegree(k);
    C.FieldOptions.Add(VariableNames.GravityZ + "#A", new FieldOpts() {
        SaveToDB = FieldOpts.SaveToDBOpt.TRUE
    });
    C.FieldOptions.Add(VariableNames.GravityZ + "#B", new FieldOpts() {
        SaveToDB = FieldOpts.SaveToDBOpt.TRUE
    });

    // physical parameters
    C.PhysicalParameters.rho_A = rhoA;
    C.PhysicalParameters.mu_A = muA;
    
    C.PhysicalParameters.rho_B = rhoB;
    C.PhysicalParameters.mu_B = muB;
    
    C.PhysicalParameters.Sigma = sigma;
    
    C.PhysicalParameters.IncludeConvection = true;

    
    if (restartRun) {
        C.RestartInfo = new Tuple<Guid, BoSSS.Foundation.IO.TimestepNumber>(restartSession.ID, restartTS);
    } else {
        // set grid
        C.SetGrid(Grids[idx]);
    
        // initial conditions   
        C.AddInitialValue("Phi", PhiFunc);   
         
        C.AddInitialValue("GravityZ#A", GravityValue);
        C.AddInitialValue("GravityZ#B", GravityValue);
    }
    
    C.AdvancedDiscretizationOptions.SST_isotropicMode = SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;
    C.LSContiProjectionMethod = ContinuityProjectionOption.ConstrainedDG;
 
    C.ReInitPeriod = 30;

    // C.SkipSolveAndEvaluateResidual = true;

    //C.NonLinearSolver.SolverCode = NonLinearSolverCode.Picard;
    //C.NonLinearSolver.ConvergenceCriterion = 1e-9;
    C.NonLinearSolver.MaxSolverIterations = 50;

    // C.LinearSolver = LinearSolverCode.exp_Kcycle_schwarz.GetConfig();


    C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
    C.Timestepper_LevelSetHandling = LevelSetHandling.Coupled_Once;
    C.Option_LevelSetEvolution = LevelSetEvolution.StokesExtension;
    C.TimeSteppingScheme = TimeSteppingScheme.BDF3;
    C.dtFixed = dt;
    C.NoOfTimesteps = numTs;

    if (AMRlevel > 0) {
        C.AdaptiveMeshRefinement = true;
        C.activeAMRlevelIndicators.Add(new AMRonNarrowband() { maxRefinementLevel = AMRlevel });
        C.AMR_startUpSweeps = AMRlevel;
    }

    // C.PostprocessingModules.Add(new EnergyLogging());
    C.TracingNamespaces = "BoSSS.Solution.AdvancedSolvers";
   
    if (restartRun) {
        C.SessionName = restartName;
    } else {
        int res = Resolutions[idx];
        C.SessionName = $"RB3D_fullDomain_{res}x{res}x{2*res}AMR{AMRlevel}_k{k}_Reinit30_BDF3_withOutputs_{testcase}";
        //C.SessionName = $"RB3D_quarterDomain_{res}x{res}x{2*res}AMR{AMRlevel}_k{k}_{testcase}_setttingsCheck";
    }
    
    // C.GridPartType = GridPartType.Predefined;
    // C.GridPartOptions = "FourProcSplit_fullDomain";
    
    Controls.Add(C);
}

In [ ]:
Controls.Select(C => C.SessionName)

index,value
0,RB3D_fullDomain_8x8x16AMR0_k3_Reinit30_BDF3_testcase1_withOutputs_restart1


## Launch Jobs

In [ ]:
foreach(var ctrl in Controls) {
    var oneJob              = ctrl.CreateJob();
    
    oneJob.NumberOfMPIProcs = 4;

    int numThreads = 4;
    oneJob.NumberOfThreads = numThreads;
    IDictionary<string, string> args = oneJob.EnvironmentVars;
    args["BOSSS_ARG_7"] = numThreads.ToString();
    args.Remove("OMP_NUM_THREADS");
    oneJob.MySetCommandLineArguments(args.Values.ToArray());

    //oneJob.UseComputeNodesExclusive = true;\n",

    //((SlurmClient)myBatch).ExecutionTime = "24:00:00";

    oneJob.Activate(myBatch);
}

 ------------ MSHPC FailedOrCanceled; original Failed
Deployments so far (2): (Job token: 94189, FailedOrCanceled 'RisingBubble3D-XNSE_Solver2024Sep10_150416.420415' @ MS HPC client  FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries, FailedOrCanceled), (Job token: 94190, FailedOrCanceled 'RisingBubble3D-XNSE_Solver2024Sep10_152401.950284' @ MS HPC client  FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries, FailedOrCanceled);
Success: 0
job submit count: 2
Note: Job was deployed (2) number of times, all failed; RetryCount is 3, so try once more.
Hint: want to re-activate the job.
Job is FailedOrCanceled, but retry count is set to 3 and only 2 tries yet - trying once more...
Deploying job RB3D_fullDomain_8x8x16AMR0_k3_Reinit30_BDF3_testcase1_withOutputs_restart1 ... 
Set Database: { Session Count = 8; Grid Count = 4; Path = \\dc3\userspace\smuda\hpccluster\RisingBubble3D }
Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\smud